# 04 — Token Tax Lab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/marcoharuni/forge-tokenizer/blob/main/notebooks/04_token_tax_lab.ipynb)

Measure how the same local tokenizer treats languages and formats differently.

**Runtime:** CPU is fine. No GPU required.

---

## 1. Install & Clone

In [ ]:
%pip install -q numpy matplotlib pytest tqdm regex
%cd /content
!test -d forge-tokenizer || git clone https://github.com/marcoharuni/forge-tokenizer.git
%cd /content/forge-tokenizer
!git pull --ff-only
%pip install -q -e .

import sys
src = '/content/forge-tokenizer/src'
if src not in sys.path:
    sys.path.insert(0, src)
print('Ready. Run the Imports cell next.')

## 2. Imports

In [ ]:
from pathlib import Path
assert Path('src/forge_tokenizer').exists(), 'Run section 1: Install & Clone first.'
import json
import os
os.environ.setdefault('MPLCONFIGDIR', 'generated/.mplconfig')
Path('generated/.mplconfig').mkdir(parents=True, exist_ok=True)
import matplotlib.pyplot as plt

from forge_tokenizer import ForgeTokenizer
from forge_tokenizer.metrics import compression_ratio, count_words, fertility, single_token_retention_rate

print('forge-tokenizer ready')

## 3. Train The Local Educational Tokenizer

These numbers depend on the tiny local corpus. They are learning measurements, not universal claims.

In [ ]:
corpus = Path('data/tiny_corpus.txt').read_text(encoding='utf-8')
tokenizer = ForgeTokenizer().train(corpus, num_merges=100)
print('vocab size:', tokenizer.vocab_size)

## 4. Measure Languages And Code

In [ ]:
samples = json.loads(Path('data/multilingual_samples.json').read_text(encoding='utf-8'))
rows = []
for name, text in samples.items():
    ids = tokenizer.encode(text)
    rows.append({
        'name': name,
        'characters': len(text),
        'words': count_words(text),
        'tokens': len(ids),
        'fertility': fertility(tokenizer, text),
        'compression': compression_ratio(tokenizer, text),
        'single_token_retention': single_token_retention_rate(tokenizer, text),
    })

for row in rows:
    print(row)

## 5. Cost Multiplier Versus English

A simple proxy: if a sample needs twice as many tokens per word as English, it has a 2x token-count multiplier in this local measurement.

In [ ]:
english = next(row for row in rows if row['name'] == 'English')
for row in rows:
    row['multiplier_vs_english'] = row['fertility'] / english['fertility'] if english['fertility'] else 0.0
    print(f"{row['name']:12s} x{row['multiplier_vs_english']:.2f}")

## 6. Plot The Local Token Tax

In [ ]:
labels = [row['name'] for row in rows]
values = [row['multiplier_vs_english'] for row in rows]

fig, ax = plt.subplots(figsize=(8, 4.5))
bars = ax.bar(labels, values, color='#2563eb', edgecolor='#111827', linewidth=0.8)
ax.axhline(1.0, color='#111827', linewidth=1, linestyle='--')
ax.set_title('Local token tax multiplier vs English')
ax.set_ylabel('Multiplier')
ax.tick_params(axis='x', rotation=25)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(True, axis='y', alpha=0.22)
for bar, value in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width() / 2, value, f'{value:.2f}x', ha='center', va='bottom', fontsize=8)
plt.tight_layout()
plt.show()

---

**Next →** [05 Token ID To Meaning](05_token_id_to_meaning.ipynb)